In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/khaidphan/rcnn-resnet50/pytorch/default/1/best_model.pth


In [2]:
!ls /kaggle/input/models/khaidphan/rcnn-resnet50/pytorch/default/1

best_model.pth


In [3]:
### Download dataset
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="9LrAPmXuhWAvoilYVxbM")
project = rf.workspace("eagle-eye").project("basketball-1zhpe")
version = project.version(1)
dataset = version.download("coco")
                

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 102.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0,


Extracting Dataset Version Zip to Basketball-1 in coco:: 100%|██████████| 2607/2607 [00:00<00:00, 7823.18it/s]


In [4]:
### Installing packages / dependencies


In [5]:
### Importing

import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import os
from pathlib import Path

import json
from collections import defaultdict
 

In [6]:
### Model Setup
 
def build_model(num_classes: int, pretrained: bool = True) -> torch.nn.Module:
    
    # Loads pre-trained weights and model
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
    model = fasterrcnn_resnet50_fpn(weights=weights)
 
    # Replace the classification head to match your class count
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
 
    return model

In [7]:
### Dataset

BASKETBALL_CATEGORY_ID = 1 
 
 
class MyDataset(Dataset):
 
    def __init__(self, split_dir: str, transforms=None):
        self.split_dir = split_dir
        self.transforms = transforms
 
        ann_path = os.path.join(split_dir, "_annotations.coco.json")
        with open(ann_path) as f:
            coco = json.load(f)
 
        # Map image_id to file_name so we can load the right file
        self.images = {img["id"]: img["file_name"] for img in coco["images"]}
 
        # Group annotations by image_id, keeping only category_id=1
        self.annotations = defaultdict(list)
        for ann in coco["annotations"]:
            if ann["category_id"] == BASKETBALL_CATEGORY_ID:
                self.annotations[ann["image_id"]].append(ann)
 
        # Ordered list of image ids so __getitem__ has a stable index
        self.image_ids = list(self.images.keys())
 
    def __len__(self):
        return len(self.image_ids)
 
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        file_name = self.images[image_id]
 
        # Load image
        img_path = os.path.join(self.split_dir, file_name)
        image = Image.open(img_path).convert("RGB")
 
        # Build target
        boxes, labels = [], []
        for ann in self.annotations[image_id]:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x + w, y + h])     # xywh => xyxy
            labels.append(1)                       
 
        if boxes:
            target = {
                "boxes":  torch.as_tensor(boxes,  dtype=torch.float32),
                "labels": torch.as_tensor(labels, dtype=torch.int64),
            }
        else:
            # Image has no basketball annotations, return empty tensors
            target = {
                "boxes":  torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,),   dtype=torch.int64),
            }
 
        if self.transforms:
            image = self.transforms(image)
 
        return image, target
 

 
def get_transforms(train: bool):
    tfms = [T.ToTensor()]
    if train:
        tfms.append(T.RandomHorizontalFlip(p=0.5))
    return T.Compose(tfms)
 
 
def collate_fn(batch):
    return tuple(zip(*batch))
 


In [8]:
### Training


def train_one_epoch(model, optimizer, loader, device, epoch):
    model.train()
    total_loss = 0.0
 
    for i, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
 
        # Forward pass returns a dict of losses when model.train()
        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())
 
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
 
        total_loss += losses.item()
 
        if i % 20 == 0:
            print(
                f"Epoch {epoch} | step {i}/{len(loader)} | "
                f"loss: {losses.item():.4f} | "
                + " | ".join(f"{k}: {v.item():.4f}" for k, v in loss_dict.items())
            )
 
    return total_loss / len(loader)
 
 
def train(data_root: str = "/kaggle/working/Basketball-1", num_classes: int = 2, num_epochs: int = 10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
 
    # Datasets & loaders
    train_ds = MyDataset(os.path.join(data_root, "train"), get_transforms(train=True))
    val_ds   = MyDataset(os.path.join(data_root, "valid"),   get_transforms(train=False))
 
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=4)
    val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=4)
 
    # Model
    model = build_model(num_classes, pretrained=True).to(device)
 
    # Freeze backbone stages 1 & 2, fine-tune from stage 3 onward
    for name, param in model.backbone.named_parameters():
        if "layer1" in name or "layer2" in name or "body.conv1" in name or "body.bn1" in name:
            param.requires_grad = False
 
    # Optimizer — separate LR for backbone vs head
    params = [
        {"params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad], "lr": 1e-4},
        {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad], "lr": 5e-4},
    ]
    optimizer = torch.optim.AdamW(params, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
 
    best_loss = float("inf")
    for epoch in range(1, num_epochs + 1):
        avg_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
        scheduler.step()
        print(f"\nEpoch {epoch} avg loss: {avg_loss:.4f}\n")
 
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
            print("Saved best model.")
 
    return model

In [9]:
### Inference

def load_model(checkpoint_path: str, num_classes: int, device) -> torch.nn.Module:
    model = build_model(num_classes, pretrained=False)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device).eval()
    return model
 
 
def preprocess_image(image_path: str) -> tuple[torch.Tensor, tuple]:
    """Load and convert image to float tensor in [0, 1]."""
    image = Image.open(image_path).convert("RGB")
    original_size = image.size  # (W, H)
    tensor = T.ToTensor()(image)  # [3, H, W], values in [0, 1]
    return tensor, original_size
 
 
@torch.inference_mode()
def predict(
    model: torch.nn.Module,
    image_path: str = "/kaggle/working/Basketball-1/test",
    device='cpu',
    score_threshold: float = 0.5,
    class_names: list[str] | None = None,
) -> list[dict]:
    tensor, _ = preprocess_image(image_path)
    tensor = tensor.to(device)
 
    # model expects a list of tensors
    outputs = model([tensor])[0]
 
    results = []
    for box, label, score in zip(outputs["boxes"], outputs["labels"], outputs["scores"]):
        if score.item() < score_threshold:
            continue
        entry = {
            "path": str(image_path),
            "box":   [round(v, 1) for v in box.cpu().tolist()],
            "label": label.item(),
            "score": round(score.item(), 4),
        }
        if class_names:
            entry["class"] = class_names[label.item()]
        results.append(entry)
 
    return results
 
 
@torch.inference_mode()
def predict_batch(
    model: torch.nn.Module,
    image_paths: list[str],
    device,
    score_threshold: float = 0.25,
) -> list[list[dict]]:
    tensors = [T.ToTensor()(Image.open(p).convert("RGB")).to(device) for p in image_paths]

    print(f"Tensor length: {len(tensors)}")
    batch_outputs = model(tensors)
 
    all_results = []
    for outputs, image_path in zip(batch_outputs, image_paths):
        results = {
            'path': str(image_path),
            'detections': []
        }
        for box, label, score in zip(outputs["boxes"], outputs["labels"], outputs["scores"]):
            if score.item() >= score_threshold:
                results['detections'].append({
                    "box":   [round(v, 1) for v in box.cpu().tolist()],
                    "label": label.item(),
                    "score": round(score.item(), 4),
                })
        all_results.append(results)
 
    return all_results
 
 
 
def demo_coco_pretrained(image_path: str="/kaggle/working/Basketball-1/test", score_threshold: float = 0.25, num_pred: int = -1):
    COCO_CLASSES = [
        "__background__", "basketball",
    ]
 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    model = load_model("/kaggle/input/models/khaidphan/rcnn-resnet50/pytorch/default/1/best_model.pth",
      2,
      device,
    )
    jpg_files = list(Path(image_path).rglob('*.jpg'))
    if num_pred != -1 and num_pred > 0: 
        jpg_files = jpg_files[0:num_pred]
        
    
    # results = predict(model, jpg_files, device, score_threshold, COCO_CLASSES)
    results = predict_batch(model, jpg_files, device, score_threshold) #, COCO_CLASSES)
    
    # print(f"\nDetected {len(results)} object(s) in '{image_path}':")
    # for r in results:
    #     print(r)
    #     # label = r.get("class", r["label"])
    #     # print(f"  {label:<20} score={r['score']:.3f}  box={r['box']}")
 
    return results
 

In [10]:
import cv2
import matplotlib.pyplot as plt

def show_bbox(result) -> None:
    img = cv2.imread(result['path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for dec in result['detections']: 
        x1, y1, x2, y2 = map(int, dec['box'])
        
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        label = f"{dec['label']} ({dec['score']:.2f})"
        cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    # if result['detections']: 
    #     dec = result['detections'][0]
    #     x1, y1, x2, y2 = map(int, dec['box'])
        
    #     cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
    #     label = f"{dec['label']} ({dec['score']:.2f})"
    #     cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    
    
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [11]:
### Entry point 

# if True: 
#     # Usage: python rcnn_resnet50.py path/to/image.jpg
#     all_res = demo_coco_pretrained(num_pred = 20)

#     for r in all_res: 
#         show_bbox(r)
# else:
#     print("Usage:")
#     print("  # Quick demo on COCO pretrained weights:")
#     print("  python rcnn_resnet50.py path/to/image.jpg")
#     print()
#     print("  # Fine-tune on your own data:")
    
#     train(num_classes=2, num_epochs=20)
 

In [12]:
CHECKPOINT   = "/kaggle/input/models/khaidphan/rcnn-resnet50/pytorch/default/1/best_model.pth"
DATA_ROOT    = "/kaggle/working/Basketball-1"
NUM_CLASSES  = 2
IOU_THRESH   = 0.5      # IoU threshold to count a detection as a true positive
SCORE_THRESH = 0.3      # minimum confidence to keep a prediction
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [13]:

 
 
# IoU 
 
def box_iou(box_a: torch.Tensor, box_b: torch.Tensor) -> torch.Tensor:
    """
    Compute pairwise IoU between two sets of boxes.
    box_a: [N, 4]  box_b: [M, 4]  →  returns [N, M]
    Boxes are in xyxy format.
    """
    area_a = (box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1])  # [N]
    area_b = (box_b[:, 2] - box_b[:, 0]) * (box_b[:, 3] - box_b[:, 1])  # [M]
 
    inter_x1 = torch.max(box_a[:, None, 0], box_b[None, :, 0])
    inter_y1 = torch.max(box_a[:, None, 1], box_b[None, :, 1])
    inter_x2 = torch.min(box_a[:, None, 2], box_b[None, :, 2])
    inter_y2 = torch.min(box_a[:, None, 3], box_b[None, :, 3])
 
    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter   = inter_w * inter_h                                           # [N, M]
 
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union.clamp(min=1e-6)
 
 
# Evaluation
 
def evaluate_image(pred_boxes, pred_scores, gt_boxes, iou_thresh=IOU_THRESH):
    """
    Match predictions to ground-truth boxes for one image.
 
    Returns:
        tp         : number of true positives
        fp         : number of false positives
        fn         : number of false negatives (missed gt boxes)
        ious       : IoU of each matched TP pair
        scores     : confidence score of each TP prediction
        raw        : list of dicts with full per-box detail
    """
    raw = []
 
    if len(gt_boxes) == 0 and len(pred_boxes) == 0:
        return 0, 0, 0, [], [], raw
 
    if len(pred_boxes) == 0:
        for gt in gt_boxes.tolist():
            raw.append({"gt_box": gt, "pred_box": None, "iou": 0.0,
                        "score": None, "match": "fn"})
        return 0, 0, len(gt_boxes), [], [], raw
 
    if len(gt_boxes) == 0:
        for pred, score in zip(pred_boxes.tolist(), pred_scores.tolist()):
            raw.append({"gt_box": None, "pred_box": pred, "iou": 0.0,
                        "score": score, "match": "fp"})
        return 0, len(pred_boxes), 0, [], [], raw
 
    iou_matrix = box_iou(pred_boxes, gt_boxes)   # [N_pred, N_gt]
 
    matched_gt   = set()
    matched_pred = set()
    tp_ious      = []
    tp_scores    = []
 
    # Greedy match: sort predictions by score descending
    order = pred_scores.argsort(descending=True)
    for pred_idx in order.tolist():
        best_iou, best_gt = iou_matrix[pred_idx].max(0)
        best_iou  = best_iou.item()
        best_gt   = best_gt.item()
        if best_iou >= iou_thresh and best_gt not in matched_gt:
            matched_gt.add(best_gt)
            matched_pred.add(pred_idx)
            tp_ious.append(best_iou)
            tp_scores.append(pred_scores[pred_idx].item())
            raw.append({
                "gt_box":   gt_boxes[best_gt].tolist(),
                "pred_box": pred_boxes[pred_idx].tolist(),
                "iou":      round(best_iou, 4),
                "score":    round(pred_scores[pred_idx].item(), 4),
                "match":    "tp",
            })
 
    for pred_idx in range(len(pred_boxes)):
        if pred_idx not in matched_pred:
            raw.append({
                "gt_box":   None,
                "pred_box": pred_boxes[pred_idx].tolist(),
                "iou":      0.0,
                "score":    round(pred_scores[pred_idx].item(), 4),
                "match":    "fp",
            })
 
    for gt_idx in range(len(gt_boxes)):
        if gt_idx not in matched_gt:
            raw.append({
                "gt_box":   gt_boxes[gt_idx].tolist(),
                "pred_box": None,
                "iou":      0.0,
                "score":    None,
                "match":    "fn",
            })
 
    tp = len(matched_pred)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes)   - len(matched_gt)
 
    return tp, fp, fn, tp_ious, tp_scores, raw

 
@torch.inference_mode()
def evaluate_split(model, split: str) -> dict:
    """
    Run evaluation on one split ('valid' or 'test').
    Returns a dict with all metrics + per-image raw results.
    """
    dataset = MyDataset(
        f"{DATA_ROOT}/{split}",
        transforms=T.Compose([T.ToTensor()])
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False,
                        collate_fn=collate_fn, num_workers=2)
 
    all_tp, all_fp, all_fn = 0, 0, 0
    all_ious, all_scores   = [], []
    per_image_raw          = []
 
    for img_idx, (images, targets) in enumerate(loader):
        images  = [img.to(DEVICE) for img in images]
        outputs = model(images)
 
        for output, target in zip(outputs, targets):
            # Filter predictions by score threshold
            keep        = output["scores"] >= SCORE_THRESH
            pred_boxes  = output["boxes"][keep].cpu()
            pred_scores = output["scores"][keep].cpu()
            gt_boxes    = target["boxes"].cpu()
 
            tp, fp, fn, ious, scores, raw = evaluate_image(
                pred_boxes, pred_scores, gt_boxes
            )
 
            all_tp     += tp
            all_fp     += fp
            all_fn     += fn
            all_ious   .extend(ious)
            all_scores .extend(scores)
 
            per_image_raw.append({
                "image_idx": img_idx,
                "tp": tp, "fp": fp, "fn": fn,
                "detections": raw,
            })
 
    # aggregate metrics
    precision    = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0.0
    recall       = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0.0
    f1           = (2 * precision * recall / (precision + recall)
                    if (precision + recall) > 0 else 0.0)
    mean_iou     = float(np.mean(all_ious))   if all_ious  else 0.0
    overall_iou  = float(np.sum(all_ious) / (all_tp + all_fp + all_fn)
                         if (all_tp + all_fp + all_fn) > 0 else 0.0)
    overall_score = float(np.mean(all_scores)) if all_scores else 0.0
 
    return {
        "split":         split,
        "num_images":    len(dataset),
        "total_tp":      all_tp,
        "total_fp":      all_fp,
        "total_fn":      all_fn,
        "overall_iou":   round(overall_iou,   4),
        "overall_score": round(overall_score,  4),
        "mean_iou":      round(mean_iou,       4),
        "precision":     round(precision,      4),
        "recall":        round(recall,         4),
        "f1_score":      round(f1,             4),
        "raw":           per_image_raw,
    }
 
  
def print_summary(results: dict):
    s = results["split"].upper()
    print(f"\n{'─'*40}")
    print(f"  {s} RESULTS  ({results['num_images']} images)")
    print(f"{'─'*40}")
    print(f"  Overall IoU    : {results['overall_iou']:.4f}")
    print(f"  Overall score  : {results['overall_score']:.4f}")
    print(f"  Mean IoU       : {results['mean_iou']:.4f}")
    print(f"  Precision      : {results['precision']:.4f}")
    print(f"  Recall         : {results['recall']:.4f}")
    print(f"  F1 score       : {results['f1_score']:.4f}")
    print(f"{'─'*40}")
    print(f"  TP: {results['total_tp']}  FP: {results['total_fp']}  FN: {results['total_fn']}")
    print(f"{'─'*40}\n")
 
 


In [14]:
model = load_model(
    "/kaggle/input/models/khaidphan/rcnn-resnet50/pytorch/default/1/best_model.pth",
    2,
    DEVICE,
)

all_results = {}
for split in ("valid", "test"):
    split_dir = f"{DATA_ROOT}/{split}"
    if not __import__("os").path.isdir(split_dir):
        print(f"Skipping '{split}' — folder not found.")
        continue

    print(f"Evaluating {split}...")
    results = evaluate_split(model, split)
    all_results[split] = results
    print_summary(results)

# with open("eval_results.json", "w") as f:
#     json.dump(all_results, f, indent=2)
# print("Raw results saved to eval_results.json")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 150MB/s]


Evaluating valid...

────────────────────────────────────────
  VALID RESULTS  (130 images)
────────────────────────────────────────
  Overall IoU    : 0.5311
  Overall score  : 0.8811
  Mean IoU       : 0.7805
  Precision      : 0.8829
  Recall         : 0.7481
  F1 score       : 0.8099
────────────────────────────────────────
  TP: 98  FP: 13  FN: 33
────────────────────────────────────────

Evaluating test...

────────────────────────────────────────
  TEST RESULTS  (472 images)
────────────────────────────────────────
  Overall IoU    : 0.6019
  Overall score  : 0.9008
  Mean IoU       : 0.8498
  Precision      : 0.8635
  Recall         : 0.7975
  F1 score       : 0.8292
────────────────────────────────────────
  TP: 386  FP: 61  FN: 98
────────────────────────────────────────

